# Position ladder v2 — exact Appendix E replication (Liu et al., Lost in the Middle)

**One row per runtime.** Set `MODEL_KEY` in cell [1]. Run order is preregistered
(`docs/position-ladder-v2-prereg-2026-08-01.md`):

1. `llama2-13b-base` → 2. `llama2-7b-chat` (**primary gate — the study can die here**)
3. `llama2-7b-base` → 4. `llama2-13b-chat` → 5–8. extensions, only after `ANCHOR_OK`.

Full 2655 questions per position, all five positions. Anchor rows run their
`get_qa_responses_from_llama_2.py` unmodified; scoring is their
`evaluate_qa_responses.py`. Adjudicator `position_ladder_qa.py` embedded
byte-identical (`088166614ed479b0…`).

Known deviation recorded per row: current vLLM instead of their 2023 pin
(`v0.2.1.post1`), greedy decoding throughout.


In [ ]:
# [1] preflight
MODEL_KEY = "llama2-13b-base"   # see position_ladder_qa.ROWS for the 8 keys
import torch, time
T0 = time.time()
assert torch.cuda.is_available(), "no GPU"
_props = torch.cuda.get_device_properties(0)
GPU_NAME, GIB = _props.name, _props.total_memory / 1024**3
print(GPU_NAME, round(GIB, 1), "GiB")
BUDGET_H = {"13b": 4.5}.get(MODEL_KEY.split("-")[1][:3], 3.0)
assert GIB >= (38 if "13b" in MODEL_KEY else 20), f"{GIB:.1f} GiB too small for {MODEL_KEY}"

def elapsed_h(): return (time.time() - T0) / 3600.0
def budget_check(where):
    e = elapsed_h()
    assert e <= BUDGET_H, (f"OVERRUN at {where}: {e:.2f}h > {BUDGET_H}h. Partial "
                           "files are persisted. Diagnose; do not raise to fit.")
    return e


In [ ]:
# [2] pinned adjudicator, self-tested on the box before any spend
import hashlib, pathlib, sys, importlib, inspect
PLQ_SHA = "088166614ed479b0ace1501e6347c5df9908466599ca8eb94ed5a770936c6887"
PLQ_SRC = '"""Position ladder v2: EXACT replication of Liu et al. Appendix E, then extension.\n\nSUPERSEDES the v1 KV instrument (position_ladder.py, kept frozen for provenance).\nPrereg: docs/position-ladder-v2-prereg-2026-08-01.md. Verdict = evaluate() here.\n\nWHY V2 EXISTS. v1 anchored on the real Appendix E claim -- "only the larger\nmodels (13B and 70B) exhibit the U-shaped performance curve ... the smallest\nLlama-2 models (7B) are solely recency-biased" -- but implemented the WRONG\nTASK. The claim is about multi-document QA (20 NQ docs, ~4K tokens); v1 built\nkey-value retrieval, which Llama-2 was never run on and whose smallest published\nsetting does not even fit Llama-2\'s window (their Table 4: 75 pairs = 5444\ntokens on a Llama tokenizer). v1\'s ANCHOR_FAILED therefore meant "wrong\ninstrument", not "result fails to reproduce".\n\nV2 runs THEIR released pipeline verbatim -- data, prompt template, chat\nformatting, sampling parameters, scorer -- pinned by commit and file hash below.\nThis module never re-implements any of it; it only adjudicates their scored\noutputs.\n\nRun:  python3 position_ladder_qa.py --self-test\n"""\nfrom __future__ import annotations\n\nfrom typing import Any\n\n# ---------------------------------------------------------------- pinned upstream\nLITM_REPO = "https://github.com/nelson-liu/lost-in-the-middle.git"\nLITM_COMMIT = "29b8a6d042ce29abccee3db1a73171a107d7e6af"\nLITM_SHAS = {\n    "src/lost_in_the_middle/prompting.py":\n        "c8de5f26e22261ed540e7578ff4ca79b7548aa5b9ddbc33b769afdae494acb26",\n    "src/lost_in_the_middle/metrics.py":\n        "b2ac7e1c2fac7e1d09aabbeee782bebe4fb74b650b42cb9111b88692e5a2f35e",\n    "src/lost_in_the_middle/prompts/qa.prompt":\n        "368a84fe24373cc0a3a789ce5e62e6854994ece138747c93c76f25aafea25411",\n    "scripts/get_qa_responses_from_llama_2.py":\n        "43506abcf9ac0ea7f98a24cef2440485bb4db3c066e6b317a3b8fbe71e53c1f8",\n    "scripts/evaluate_qa_responses.py":\n        "fd35118106f02e781143f720c21b71ceeb8e4592e5b440f54f035a8562dd5b63",\n    "qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz":\n        "69a8dd87a45f07fb3ed9b804917a89db975cd6e488682fc838edca029cc3881f",\n    "qa_data/20_total_documents/nq-open-20_total_documents_gold_at_4.jsonl.gz":\n        "c5e44700c4db24536f139f7383e3dd002413515ee889aeadc72767dd7f289682",\n    "qa_data/20_total_documents/nq-open-20_total_documents_gold_at_9.jsonl.gz":\n        "cbada411f70235afc47d9b5407e9358ad72eb021742402ff032c8b65076ffb2f",\n    "qa_data/20_total_documents/nq-open-20_total_documents_gold_at_14.jsonl.gz":\n        "86c1a25093615baefdd1ee4db7a993548b3bef24ca02eb9bd8db43fbedf517f2",\n    "qa_data/20_total_documents/nq-open-20_total_documents_gold_at_19.jsonl.gz":\n        "d79b715e1b8c8301335e93cb01659f68c24f0253a808c3190b06d21896506c6c",\n}\n\nPOSITIONS = (0, 4, 9, 14, 19)          # their gold_at indices, 20-document setting\nN_EXPECTED = 2655                      # examples per gold_at file\n# Their script skips prompts > 4096 Llama-2 tokens (~20/2655, Appendix E).\nN_MIN = 2600\n\n# protocol constants, theirs\nTEMPERATURE, TOP_P, MAX_NEW_TOKENS, MAX_PROMPT_LENGTH = 0.0, 1.0, 100, 4096\n\nROWS = {\n    # anchors: run via THEIR script, verbatim\n    "llama2-7b-base":   dict(hf="meta-llama/Llama-2-7b-hf",         role="anchor",    variant="base"),\n    "llama2-7b-chat":   dict(hf="meta-llama/Llama-2-7b-chat-hf",    role="anchor",    variant="chat"),\n    "llama2-13b-base":  dict(hf="meta-llama/Llama-2-13b-hf",        role="anchor",    variant="base"),\n    "llama2-13b-chat":  dict(hf="meta-llama/Llama-2-13b-chat-hf",   role="anchor",    variant="chat"),\n    # extensions: their prompt + scorer + sampling, own chat template (documented\n    # deviation -- their format_chat_prompt is Llama-2-specific). Base-variant ids\n    # are verified at Phase 0; they exist to de-confound tuning (see FIG16 below).\n    "llama31-8b-base":  dict(hf="meta-llama/Llama-3.1-8B",          role="extension", variant="base"),\n    "llama31-8b-chat":  dict(hf="meta-llama/Llama-3.1-8B-Instruct", role="extension", variant="chat"),\n    "olmo3-7b-base":    dict(hf="allenai/Olmo-3-1025-7B",           role="extension", variant="base"),\n    "olmo3-7b-chat":    dict(hf="allenai/Olmo-3-7B-Instruct",       role="extension", variant="chat"),\n}\n\n# Figure 16 digitized by hand from the published plot, +/-2pp reading error.\n# These are TARGETS for the anchor gate, fixed before any generation.\nFIG16 = {\n    "llama2-7b-base":  {0: 0.230, 4: 0.220, 9: 0.235, 14: 0.245, 19: 0.405},\n    "llama2-7b-chat":  {0: 0.435, 4: 0.415, 9: 0.445, 14: 0.460, 19: 0.560},\n    "llama2-13b-base": {0: 0.420, 4: 0.215, 9: 0.210, 14: 0.250, 19: 0.410},\n    "llama2-13b-chat": {0: 0.510, 4: 0.480, 9: 0.480, 14: 0.525, 19: 0.585},\n}\n\n# THE TUNING CAVEAT, from their own Figure 16: 13B base -> chat collapses the\n# primacy index from ~+0.195 to ~+0.015. Chat-tuning can SUPPRESS primacy, so an\n# extension CHAT row without primacy is evidence about nothing; only BASE rows\n# can speak to the parameter-count question.\n\nBARS = dict(\n    no_prim=0.05,   # G1: a 7B anchor "lacks primacy" if index < this\n    prim=0.10,      # G2: 13B-base "has primacy" if index >= this\n    rec=0.05,       # G3: every anchor must show recency >= this\n)\nSENS = dict(no_prim=(0.03, 0.07), prim=(0.07, 0.13), rec=(0.03, 0.07))\n\n\ndef indices(curve: dict[int, float]) -> dict[str, float]:\n    """Primacy/recency indices against the INTERIOR MEAN (positions 4, 9, 14).\n\n    Same estimator class as v1 after its min->mean fix; interior mean is\n    unbiased and pools 3x the data of any single point.\n    """\n    assert set(curve) == set(POSITIONS), f"need exactly positions {POSITIONS}"\n    interior = (curve[4] + curve[9] + curve[14]) / 3\n    return dict(primacy=round(curve[0] - interior, 4),\n                recency=round(curve[19] - interior, 4),\n                interior_mean=round(interior, 4))\n\n\ndef classify(curve: dict[int, float], bars: dict = BARS) -> dict[str, Any]:\n    ix = indices(curve)\n    p, r = ix["primacy"], ix["recency"]\n    prim = "PRESENT" if p >= bars["prim"] else \\\n           "ABSENT" if p < bars["no_prim"] else "INDETERMINATE"\n    return dict(**ix, primacy_call=prim, recency_ok=r >= bars["rec"],\n                curve={k: round(v, 4) for k, v in sorted(curve.items())})\n\n\ndef evaluate(m: dict[str, Any], bars: dict = BARS) -> dict[str, Any]:\n    """m = {row_key: {curve: {pos: acc}, n_by_pos: {pos: int}}}\n\n    Anchor gate first, built from Appendix E\'s own contrasts:\n      G1  llama2-7b-base AND llama2-7b-chat primacy < no_prim\n      G2  llama2-13b-base primacy >= prim\n      G3  every present anchor recency >= rec\n    13b-chat is recorded, never gated -- its published primacy (~+0.015) sits\n    inside measurement noise by design, so a gate on it would be undiagnostic.\n    """\n    per: dict[str, Any] = {}\n    for key, d in m.items():\n        c = d.get("curve") or {}\n        nb = d.get("n_by_pos") or {}\n        if set(c) != set(POSITIONS):\n            per[key] = dict(verdict="V_INCOMPLETE",\n                            reasons=[f"positions {sorted(c)} != {list(POSITIONS)}"])\n            continue\n        low = {p: n for p, n in nb.items() if n < N_MIN}\n        if len(nb) != len(POSITIONS) or low:\n            per[key] = dict(verdict="V_N_MISMATCH",\n                            reasons=[f"n per position below {N_MIN} or missing: "\n                                     f"{low or \'missing counts\'}"])\n            continue\n        per[key] = dict(verdict="OK", **classify(c, bars))\n\n    def ok(k):\n        return k in per and per[k]["verdict"] == "OK"\n\n    g1_rows = ("llama2-7b-base", "llama2-7b-chat")\n    reasons, failed = [], []\n    if not (ok("llama2-13b-base") and all(ok(k) for k in g1_rows)):\n        gate = dict(status="INCOMPLETE",\n                    reasons=[f"gate rows not all readable: "\n                             f"{[k for k in (*g1_rows, \'llama2-13b-base\') if not ok(k)]}"])\n    else:\n        for k in g1_rows:\n            if per[k]["primacy"] >= bars["no_prim"]:\n                failed.append(f"G1 {k}: primacy {per[k][\'primacy\']:+.3f} >= {bars[\'no_prim\']}")\n        if per["llama2-13b-base"]["primacy"] < bars["prim"]:\n            failed.append(f"G2 llama2-13b-base: primacy "\n                          f"{per[\'llama2-13b-base\'][\'primacy\']:+.3f} < {bars[\'prim\']}")\n        for k in (*g1_rows, "llama2-13b-base", "llama2-13b-chat"):\n            if ok(k) and not per[k]["recency_ok"]:\n                failed.append(f"G3 {k}: recency {per[k][\'recency\']:+.3f} < {bars[\'rec\']}")\n        gate = dict(status="ANCHOR_FAILED" if failed else "ANCHOR_OK",\n                    reasons=failed or ["Appendix E reproduced: 7B rows lack primacy, "\n                                       "13B-base shows it, recency everywhere"])\n\n    out = dict(anchor_gate=gate, per_model=per)\n    if gate["status"] != "ANCHOR_OK":\n        out["driver"] = None\n        out["reasons"] = ["instrument not validated; extension rows not interpreted"]\n        return out\n\n    # Driver logic. ONLY BASE extension rows bear on parameter count -- their own\n    # Figure 16 shows chat-tuning collapsing 13B primacy 0.195 -> 0.015, so a\n    # chat row\'s absence is never evidence about parameters.\n    ext_base = {k: per[k] for k in ("llama31-8b-base", "olmo3-7b-base") if ok(k)}\n    ext_chat = {k: per[k] for k in ("llama31-8b-chat", "olmo3-7b-chat") if ok(k)}\n    calls_b = {k: v["primacy_call"] for k, v in ext_base.items()}\n    calls_c = {k: v["primacy_call"] for k, v in ext_chat.items()}\n    out["extension_calls"] = dict(base=calls_b, chat=calls_c)\n\n    if len(ext_base) < 2:\n        out["driver"] = None\n        out["reasons"] = [f"need both base extension rows readable, have {sorted(ext_base)}"]\n    elif any(v == "PRESENT" for v in calls_b.values()):\n        out["driver"] = "NOT_PARAMETER_COUNT"\n        out["reasons"] = [f"primacy at 7-8B params in a later-generation BASE model: "\n                          f"{[k for k, v in calls_b.items() if v == \'PRESENT\']}; "\n                          f"parameter count held vs llama2-7b-base"]\n    elif all(v == "ABSENT" for v in calls_b.values()):\n        out["driver"] = "PARAMETER_COUNT_CONSISTENT"\n        out["reasons"] = ["no 7-8B base model shows primacy despite very different "\n                          "training; consistent with (not proof of) a parameter threshold"]\n    else:\n        out["driver"] = "UNRESOLVED"\n        out["reasons"] = [f"indeterminate base calls: {calls_b}"]\n    return out\n\n\ndef sensitivity(m: dict[str, Any]) -> dict[str, Any]:\n    base = evaluate(m)\n    bk = (base["anchor_gate"]["status"], base.get("driver"))\n    grid = {}\n    for key, vals in SENS.items():\n        for v in vals:\n            b = dict(BARS); b[key] = v\n            r = evaluate(m, b)\n            grid[f"{key}={v}"] = (r["anchor_gate"]["status"], r.get("driver"))\n    return dict(base=bk, grid=grid, fragile=any(x != bk for x in grid.values()))\n\n\ndef _selftest() -> None:\n    full_n = {p: 2635 for p in POSITIONS}\n\n    def row(curve, n_by_pos=None):\n        return dict(curve=dict(curve), n_by_pos=n_by_pos or dict(full_n))\n\n    # 1. the digitized published curves must PASS the gate -- if the gate cannot\n    #    accept the paper\'s own figure, the gate is wrong, not the paper\n    anchors = {k: row(v) for k, v in FIG16.items()}\n    e = evaluate(anchors)\n    assert e["anchor_gate"]["status"] == "ANCHOR_OK", e["anchor_gate"]\n    assert e["driver"] is None, "driver claimed with no extension rows"\n    ix = indices(FIG16["llama2-13b-base"])\n    assert 0.15 < ix["primacy"] < 0.25 and ix["recency"] > 0.15, ix\n    assert indices(FIG16["llama2-7b-chat"])["primacy"] < 0.01\n    assert indices(FIG16["llama2-7b-base"])["primacy"] < 0.01\n\n    # 2. swapping 7B and 13B-base curves must FAIL both G1 and G2\n    swapped = dict(anchors)\n    swapped["llama2-7b-chat"] = row(FIG16["llama2-13b-base"])\n    swapped["llama2-13b-base"] = row(FIG16["llama2-7b-chat"])\n    e2 = evaluate(swapped)\n    assert e2["anchor_gate"]["status"] == "ANCHOR_FAILED"\n    assert any("G1" in r for r in e2["anchor_gate"]["reasons"])\n    assert any("G2" in r for r in e2["anchor_gate"]["reasons"])\n    assert e2["driver"] is None\n\n    # 3. driver worlds, on top of passing anchors\n    prim_curve = FIG16["llama2-13b-base"]          # has primacy\n    flat_rec = FIG16["llama2-7b-chat"]             # recency only\n    ext = lambda b1, b2, c1=flat_rec, c2=flat_rec: {\n        **anchors,\n        "llama31-8b-base": row(b1), "olmo3-7b-base": row(b2),\n        "llama31-8b-chat": row(c1), "olmo3-7b-chat": row(c2)}\n    e3 = evaluate(ext(prim_curve, flat_rec))\n    assert e3["driver"] == "NOT_PARAMETER_COUNT", e3["reasons"]\n    e4 = evaluate(ext(flat_rec, flat_rec))\n    assert e4["driver"] == "PARAMETER_COUNT_CONSISTENT", e4["reasons"]\n    # chat-only primacy must NOT drive a parameter claim in either direction\n    e5 = evaluate(ext(flat_rec, flat_rec, c1=prim_curve))\n    assert e5["driver"] == "PARAMETER_COUNT_CONSISTENT"\n    assert e5["extension_calls"]["chat"]["llama31-8b-chat"] == "PRESENT"\n    # indeterminate base -> UNRESOLVED\n    mid = {0: 0.315, 4: 0.24, 9: 0.24, 14: 0.24, 19: 0.40}   # prim = +0.075\n    e6 = evaluate(ext(mid, flat_rec))\n    assert e6["driver"] == "UNRESOLVED", e6\n\n    # 4. void paths: missing positions / low n block everything downstream\n    broken = dict(anchors)\n    broken["llama2-13b-base"] = dict(curve={0: .4, 4: .2}, n_by_pos={0: 2635, 4: 2635})\n    e7 = evaluate(broken)\n    assert e7["per_model"]["llama2-13b-base"]["verdict"] == "V_INCOMPLETE"\n    assert e7["anchor_gate"]["status"] == "INCOMPLETE" and e7["driver"] is None\n    lown = dict(anchors)\n    lown["llama2-7b-base"] = row(FIG16["llama2-7b-base"], {p: 500 for p in POSITIONS})\n    e8 = evaluate(lown)\n    assert e8["per_model"]["llama2-7b-base"]["verdict"] == "V_N_MISMATCH"\n    assert e8["driver"] is None, "subsampled row treated as full-n evidence"\n\n    # 5. missing extension rows block the driver, never default it\n    assert evaluate({**anchors, "llama31-8b-base": row(prim_curve)})["driver"] is None\n\n    s = sensitivity({**anchors,\n                     "llama31-8b-base": row(prim_curve), "olmo3-7b-base": row(flat_rec),\n                     "llama31-8b-chat": row(flat_rec), "olmo3-7b-chat": row(flat_rec)})\n    assert s["base"] == ("ANCHOR_OK", "NOT_PARAMETER_COUNT")\n    print("self-test OK: published curves pass, swapped curves fail G1+G2, "\n          "4 driver worlds, chat rows inert, 2 void paths, subsample refused")\n    print(f"  fig16 indices: 13b-base prim {indices(FIG16[\'llama2-13b-base\'])[\'primacy\']:+.3f}, "\n          f"7b-chat prim {indices(FIG16[\'llama2-7b-chat\'])[\'primacy\']:+.3f}")\n    print(f"  sensitivity fragile: {s[\'fragile\']}")\n\n\nif __name__ == "__main__":\n    import sys\n    if "--self-test" in sys.argv: _selftest()\n    else: print(__doc__)\n'
assert hashlib.sha256(PLQ_SRC.encode()).hexdigest() == PLQ_SHA, "artifact drift"
pathlib.Path("position_ladder_qa.py").write_text(PLQ_SRC)
sys.modules.pop("position_ladder_qa", None)
import position_ladder_qa as Q
importlib.reload(Q)
# behavioural staleness check, not a hash on the file
assert "llama2-13b-base" in Q.ROWS and hasattr(Q, "LITM_SHAS"), "stale module in memory"
Q._selftest()
SPEC = Q.ROWS[MODEL_KEY]
print("\nrow:", MODEL_KEY, SPEC)


In [ ]:
# [3] their pipeline, pinned by commit and file hash -- the instrument is theirs
import subprocess, hashlib, os
if not os.path.isdir("litm"):
    subprocess.run(["git", "clone", Q.LITM_REPO, "litm"], check=True)
subprocess.run(["git", "-C", "litm", "checkout", Q.LITM_COMMIT], check=True,
               capture_output=True)
for rel, want in Q.LITM_SHAS.items():
    got = hashlib.sha256(open(f"litm/{rel}", "rb").read()).hexdigest()
    assert got == want, f"UPSTREAM DRIFT {rel}: {got[:12]} != {want[:12]}"
print(f"litm @ {Q.LITM_COMMIT[:12]}: all {len(Q.LITM_SHAS)} file hashes verified")

# their package + current vllm (their v0.2.1.post1 pin will not build on this
# stack -- preregistered deviation #1; version recorded into the row summary)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "./litm",
                "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm", "xopen",
                "pydantic", "regex"], check=True)
import vllm
VLLM_VERSION = vllm.__version__
print("vllm", VLLM_VERSION, "(their pin: v0.2.1.post1 -- deviation, recorded)")


In [ ]:
# [4] persistence + PHASE 0 access gate (resolve a FILE for all 8 rows)
import io, json
from google.colab import userdata
from huggingface_hub import HfApi, create_repo, hf_hub_download

HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN            # their script reads the ambient login
api = HfApi(token=HF_TOKEN)
DS = f"{api.whoami()['name']}/phi-map-position-ladder-qa"
create_repo(DS, repo_type="dataset", private=True, exist_ok=True, token=HF_TOKEN)
print("persisting to", DS)

def persist(path, name):
    api.upload_file(path_or_fileobj=path, path_in_repo=name, repo_id=DS,
                    repo_type="dataset", token=HF_TOKEN)

def have(name):
    try:
        return hf_hub_download(DS, name, repo_type="dataset", token=HF_TOKEN)
    except Exception:
        return None

bad = []
for k, spec in Q.ROWS.items():
    try:
        hf_hub_download(spec["hf"], "config.json", token=HF_TOKEN)
        print(f"  ok    {k:16s} {spec['hf']}")
    except Exception as e:
        bad.append(k); print(f"  FAIL  {k:16s} {spec['hf']}  {type(e).__name__}")
assert not bad, f"cannot read {bad}; request access before spending anything"


In [ ]:
# [5] generate -- their runner for anchors, thin wrapper for extensions.
# Persist and resume per gold_index file.
import glob, shutil
GOLD = list(Q.POSITIONS)
OUTDIR = "preds"; os.makedirs(OUTDIR, exist_ok=True)

def outname(g):  return f"{MODEL_KEY}-gold_at_{g}-predictions.jsonl.gz"

PATCHED_LOAD_FORMAT = False
def run_their_script(g):
    """Anchor path: their script, their args, unmodified -- except the one
    preregistered environment patch (load_format) applied only on failure."""
    global PATCHED_LOAD_FORMAT
    args = [sys.executable, "-u", "litm/scripts/get_qa_responses_from_llama_2.py",
            "--input-path", f"litm/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_{g}.jsonl.gz",
            "--model", SPEC["hf"], "--max-new-tokens", "100", "--num-gpus", "1",
            "--output-path", f"{OUTDIR}/{outname(g)}"]
    r = subprocess.run(args, capture_output=True, text=True)
    if r.returncode != 0 and "load_format" in (r.stderr + r.stdout):
        # preregistered deviation #2: modern vllm rejects load_format="pt"
        s = open("litm/scripts/get_qa_responses_from_llama_2.py").read()
        open("litm/scripts/get_qa_responses_from_llama_2.py", "w").write(
            s.replace('load_format="pt",', 'load_format="auto",'))
        PATCHED_LOAD_FORMAT = True
        print("  patched load_format pt->auto (recorded), retrying")
        r = subprocess.run(args, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"their script failed on gold_at_{g}")

def run_wrapper(g):
    """Extension path: their get_qa_prompt + their sampling params + this
    model's own chat template (their format_chat_prompt is Llama-2-specific).
    Base variants use the raw prompt, exactly like their base-model path.
    Writes their output schema so their evaluator scores it unchanged."""
    import gzip
    from copy import deepcopy
    from transformers import AutoTokenizer
    from vllm import LLM as VLLM, SamplingParams
    sys.path.insert(0, "litm/src")
    from lost_in_the_middle.prompting import Document, get_qa_prompt

    global _ENGINE, _TOK
    if "_ENGINE" not in globals():
        _TOK = AutoTokenizer.from_pretrained(SPEC["hf"], token=HF_TOKEN)
        _ENGINE = VLLM(model=SPEC["hf"], trust_remote_code=True,
                       max_model_len=Q.MAX_PROMPT_LENGTH + Q.MAX_NEW_TOKENS + 64)
    examples, prompts = [], []
    src = f"litm/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_{g}.jsonl.gz"
    with gzip.open(src, "rt") as f:
        for line in f:
            ex = json.loads(line)
            docs = [Document.from_dict(dict(c)) for c in ex["ctxs"]]
            prompt = get_qa_prompt(ex["question"], docs,
                                   mention_random_ordering=False,
                                   query_aware_contextualization=False)
            if SPEC["variant"] == "chat":
                msgs = [{"role": "user", "content": prompt}]
                out = _TOK.apply_chat_template(msgs, tokenize=False,
                                               add_generation_prompt=True)
                prompt = out if isinstance(out, str) else str(out)   # v5 dict guard
            examples.append(ex); prompts.append(prompt)
    sp = SamplingParams(temperature=Q.TEMPERATURE, top_p=Q.TOP_P,
                        max_tokens=Q.MAX_NEW_TOKENS)
    outs = _ENGINE.generate(prompts, sp)
    with gzip.open(f"{OUTDIR}/{outname(g)}", "wt") as f:
        for ex, prompt, o in zip(examples, prompts, outs):
            rec = deepcopy(ex)
            rec["model_prompt"] = prompt
            rec["model_answer"] = o.outputs[0].text.strip()
            rec["model"] = SPEC["hf"]
            f.write(json.dumps(rec) + "\n")

for g in GOLD:
    if have(outname(g)):
        print(f"gold_at_{g}: already persisted, skipping"); continue
    print(f"gold_at_{g}: generating ({elapsed_h():.2f}h elapsed)")
    (run_their_script if SPEC["role"] == "anchor" else run_wrapper)(g)
    persist(f"{OUTDIR}/{outname(g)}", outname(g))
    print(f"gold_at_{g}: persisted  ({budget_check(f'gold_at_{g}'):.2f}h)")
print("generation complete")


In [ ]:
# [6] score with THEIR evaluator, build the curve, adjudicate this row
import gzip, random
curve, n_by_pos, examples_seen = {}, {}, {}
for g in GOLD:
    local = have(outname(g)) or f"{OUTDIR}/{outname(g)}"
    scored_name = outname(g).replace("predictions", "scored")
    r = subprocess.run([sys.executable, "-u", "litm/scripts/evaluate_qa_responses.py",
                        "--input-path", local, "--output-path", f"{OUTDIR}/{scored_name}"],
                       capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-2000:]
    rows = []
    with gzip.open(f"{OUTDIR}/{scored_name}", "rt") as f:
        rows = [json.loads(l) for l in f]
    metric_key = [k for k in rows[0] if "best_subspan_em" in k][0]
    curve[g] = sum(x[metric_key] for x in rows) / len(rows)
    n_by_pos[g] = len(rows)
    examples_seen[g] = rows
    persist(f"{OUTDIR}/{scored_name}", scored_name)
    print(f"gold_at_{g}: n={len(rows)}  best_subspan_em={curve[g]:.4f}")

result = dict(curve=curve, n_by_pos=n_by_pos)
verdict = Q.evaluate({MODEL_KEY: result})["per_model"][MODEL_KEY]
print("\nindices:", json.dumps(verdict, indent=1))
if MODEL_KEY in Q.FIG16:
    tgt = Q.indices(Q.FIG16[MODEL_KEY])
    print(f"\nFigure 16 target (±2pp digitization): primacy {tgt['primacy']:+.3f} "
          f"recency {tgt['recency']:+.3f}")


In [ ]:
# [7] hand verification + persist row summary
print("=" * 78, "\nHAND VERIFICATION: 3 answers per outcome, random draw\n", "=" * 78)
rng = random.Random(0)
for g in (GOLD[0], GOLD[-1]):
    rows = examples_seen[g]
    metric_key = [k for k in rows[0] if "best_subspan_em" in k][0]
    for label, want in (("correct", 1.0), ("wrong", 0.0)):
        pick = [x for x in rows if x[metric_key] == want]
        for x in rng.sample(pick, min(3, len(pick))):
            print(f"\n--- gold_at_{g} scored {label}")
            print("    q     :", x["question"][:100])
            print("    gold  :", x["answers"][:3])
            print("    answer:", repr(x["model_answer"][:200]))

summary = dict(protocol="POSITION_LADDER_QA_V2", model_key=MODEL_KEY, hf=SPEC["hf"],
               role=SPEC["role"], variant=SPEC["variant"],
               litm_commit=Q.LITM_COMMIT, plq_sha=PLQ_SHA,
               vllm_version=VLLM_VERSION, patched_load_format=PATCHED_LOAD_FORMAT,
               temperature=Q.TEMPERATURE, top_p=Q.TOP_P,
               max_new_tokens=Q.MAX_NEW_TOKENS, max_prompt_length=Q.MAX_PROMPT_LENGTH,
               result=result, verdict=verdict,
               wall_h=round(elapsed_h(), 3), gpu=GPU_NAME)
blob = json.dumps(summary, indent=1).encode()
open(f"row_{MODEL_KEY}.json", "wb").write(blob)
persist(f"row_{MODEL_KEY}.json", f"row_{MODEL_KEY}.json")
print("\n=== ROW SUMMARY BEGIN ===")
print(json.dumps({k: v for k, v in summary.items() if k != "result"}, indent=1))
print(json.dumps(result))
print("=== ROW SUMMARY END ===")
print("\nNext row per the preregistered order; fresh runtime.")
print("All rows done -> python3 position_ladder_qa_aggregate.py row_*.json")
